# Backpropagation: Complete Mathematical Treatment

**Author**: AI Mathematics Study Group  
**Level**: Graduate  
**Prerequisites**: Calculus (chain rule), Linear Algebra, Python  
**Estimated Time**: 3-4 hours

---

## 📚 Learning Objectives

By the end of this notebook, you will:

1. ✅ Understand computational graphs and automatic differentiation
2. ✅ Derive backpropagation from first principles using chain rule
3. ✅ Compute gradients for various layer types:
   - Fully connected (Dense)
   - Convolutional (Conv2D)
   - Batch Normalization
   - Dropout
   - Attention
4. ✅ Implement backpropagation from scratch
5. ✅ Verify gradients using numerical differentiation
6. ✅ Understand common pitfalls and debugging techniques

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Callable
from dataclasses import dataclass

np.random.seed(42)
plt.style.use('seaborn-v0_8')

print("✅ Libraries imported")
print(f"NumPy version: {np.__version__}")

## 1. The Chain Rule: Foundation of Backpropagation

### 1.1 Univariate Chain Rule

For composite function $f(g(x))$:

$$\frac{df}{dx} = \frac{df}{dg} \cdot \frac{dg}{dx}$$

**Example**:
$$f(x) = \sin(x^2)$$

Let $g(x) = x^2$, then $f(g) = \sin(g)$:

$$\frac{df}{dx} = \cos(g) \cdot 2x = 2x\cos(x^2)$$

### 1.2 Multivariate Chain Rule

For $z = f(y_1, \ldots, y_n)$ where $y_i = g_i(x_1, \ldots, x_m)$:

$$\frac{\partial z}{\partial x_j} = \sum_{i=1}^{n} \frac{\partial z}{\partial y_i} \frac{\partial y_i}{\partial x_j}$$

**This is the core of backpropagation!**

### 1.3 Vector-Valued Functions

For $\mathbf{y} = f(\mathbf{x})$ where $\mathbf{x} \in \mathbb{R}^n$, $\mathbf{y} \in \mathbb{R}^m$:

**Jacobian matrix**:
$$J = \begin{bmatrix}
\frac{\partial y_1}{\partial x_1} & \cdots & \frac{\partial y_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\
\frac{\partial y_m}{\partial x_1} & \cdots & \frac{\partial y_m}{\partial x_n}
\end{bmatrix}$$

For scalar loss $L = L(\mathbf{y})$:
$$\frac{\partial L}{\partial \mathbf{x}} = \left(\frac{\partial \mathbf{y}}{\partial \mathbf{x}}\right)^T \frac{\partial L}{\partial \mathbf{y}} = J^T \nabla_{\mathbf{y}} L$$

In [ ]:
# Example 1: Chain Rule in Action
print("Example 1: Univariate Chain Rule\n" + "="*50)

def f(x):
    """f(x) = sin(x^2)"""
    return np.sin(x**2)

def df_dx_analytical(x):
    """Analytical derivative: 2x * cos(x^2)"""
    return 2 * x * np.cos(x**2)

def df_dx_numerical(x, eps=1e-5):
    """Numerical derivative using finite differences"""
    return (f(x + eps) - f(x - eps)) / (2 * eps)

# Test
x_test = 2.0
analytical = df_dx_analytical(x_test)
numerical = df_dx_numerical(x_test)

print(f"x = {x_test}")
print(f"Analytical derivative: {analytical:.10f}")
print(f"Numerical derivative:  {numerical:.10f}")
print(f"Difference: {abs(analytical - numerical):.2e}")

# Visualize
x = np.linspace(-3, 3, 200)
y = f(x)
dy_dx = df_dx_analytical(x)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(x, y, linewidth=2, label='$f(x) = \\sin(x^2)$')
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('f(x)', fontsize=12)
ax1.set_title('Function', fontsize=14)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)

ax2.plot(x, dy_dx, linewidth=2, color='red', label="$f'(x) = 2x\\cos(x^2)$")
ax2.set_xlabel('x', fontsize=12)
ax2.set_ylabel("f'(x)", fontsize=12)
ax2.set_title('Derivative', fontsize=14)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("\n✅ Chain rule verified!")

## 2. Computational Graphs

### 2.1 What is a Computational Graph?

A **computational graph** is a directed acyclic graph (DAG) where:
- **Nodes**: Variables or operations
- **Edges**: Data flow

**Example**: $L = (w_1 x_1 + w_2 x_2 + b)^2$

```
x₁ ──┐
     ├──×── a₁ ──┐
w₁ ──┘           │
                 ├──+── z ──┐
x₂ ──┐           │         │
     ├──×── a₂ ──┤         ├──²── L
w₂ ──┘           │         │
                 │         │
b ───────────────┘         │
```

### 2.2 Forward Pass

Compute values from inputs to outputs:

1. $a_1 = w_1 x_1$
2. $a_2 = w_2 x_2$
3. $z = a_1 + a_2 + b$
4. $L = z^2$

### 2.3 Backward Pass

Compute gradients from outputs to inputs using chain rule:

$$\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial z} \frac{\partial z}{\partial a_1} \frac{\partial a_1}{\partial w_1}$$

**Step by step**:
1. $\frac{\partial L}{\partial z} = 2z$
2. $\frac{\partial z}{\partial a_1} = 1$
3. $\frac{\partial a_1}{\partial w_1} = x_1$
4. $\frac{\partial L}{\partial w_1} = 2z \cdot 1 \cdot x_1 = 2zx_1$

Similarly:
- $\frac{\partial L}{\partial w_2} = 2zx_2$
- $\frac{\partial L}{\partial b} = 2z$

In [ ]:
# Example 2: Manual Backpropagation
print("Example 2: Manual Backpropagation\n" + "="*50)

# Forward pass
x1, x2 = 2.0, 3.0
w1, w2 = 0.5, -1.0
b = 1.0

print("Forward Pass:")
a1 = w1 * x1
print(f"  a1 = w1 * x1 = {w1} * {x1} = {a1}")

a2 = w2 * x2
print(f"  a2 = w2 * x2 = {w2} * {x2} = {a2}")

z = a1 + a2 + b
print(f"  z = a1 + a2 + b = {a1} + {a2} + {b} = {z}")

L = z**2
print(f"  L = z^2 = {z}^2 = {L}")

print("\nBackward Pass:")

# dL/dz
dL_dz = 2 * z
print(f"  dL/dz = 2z = 2 * {z} = {dL_dz}")

# dL/db = dL/dz * dz/db = dL/dz * 1
dL_db = dL_dz * 1
print(f"  dL/db = dL/dz * 1 = {dL_db}")

# dL/da1 = dL/dz * dz/da1 = dL/dz * 1
dL_da1 = dL_dz * 1

# dL/da2 = dL/dz * dz/da2 = dL/dz * 1
dL_da2 = dL_dz * 1

# dL/dw1 = dL/da1 * da1/dw1 = dL/da1 * x1
dL_dw1 = dL_da1 * x1
print(f"  dL/dw1 = dL/da1 * x1 = {dL_da1} * {x1} = {dL_dw1}")

# dL/dw2 = dL/da2 * da2/dw2 = dL/da2 * x2
dL_dw2 = dL_da2 * x2
print(f"  dL/dw2 = dL/da2 * x2 = {dL_da2} * {x2} = {dL_dw2}")

print("\n" + "="*50)
print("Gradients:")
print(f"  ∂L/∂w1 = {dL_dw1}")
print(f"  ∂L/∂w2 = {dL_dw2}")
print(f"  ∂L/∂b  = {dL_db}")

# Verify with numerical gradients
eps = 1e-5

def compute_loss(w1, w2, b):
    z = w1 * x1 + w2 * x2 + b
    return z**2

dL_dw1_num = (compute_loss(w1 + eps, w2, b) - compute_loss(w1 - eps, w2, b)) / (2 * eps)
dL_dw2_num = (compute_loss(w1, w2 + eps, b) - compute_loss(w1, w2 - eps, b)) / (2 * eps)
dL_db_num = (compute_loss(w1, w2, b + eps) - compute_loss(w1, w2, b - eps)) / (2 * eps)

print("\nNumerical Verification:")
print(f"  ∂L/∂w1 (numerical) = {dL_dw1_num:.10f}")
print(f"  ∂L/∂w2 (numerical) = {dL_dw2_num:.10f}")
print(f"  ∂L/∂b  (numerical) = {dL_db_num:.10f}")

print("\nErrors:")
print(f"  |analytical - numerical| for w1: {abs(dL_dw1 - dL_dw1_num):.2e}")
print(f"  |analytical - numerical| for w2: {abs(dL_dw2 - dL_dw2_num):.2e}")
print(f"  |analytical - numerical| for b:  {abs(dL_db - dL_db_num):.2e}")

print("\n✅ Backpropagation verified!")

## 3. Neural Network: Complete Derivation

### 3.1 Network Architecture

Consider a 3-layer network:

**Layer 1** (Input → Hidden 1):
$$\begin{aligned}
z^{(1)} &= W^{(1)}x + b^{(1)} \\
a^{(1)} &= \sigma(z^{(1)})
\end{aligned}$$

**Layer 2** (Hidden 1 → Hidden 2):
$$\begin{aligned}
z^{(2)} &= W^{(2)}a^{(1)} + b^{(2)} \\
a^{(2)} &= \sigma(z^{(2)})
\end{aligned}$$

**Layer 3** (Hidden 2 → Output):
$$\begin{aligned}
z^{(3)} &= W^{(3)}a^{(2)} + b^{(3)} \\
\hat{y} &= \text{softmax}(z^{(3)})
\end{aligned}$$

**Loss** (Cross-Entropy):
$$L = -\sum_{i} y_i \log \hat{y}_i$$

### 3.2 Backpropagation Equations

Define **error term** for layer $\ell$:
$$\delta^{(\ell)} = \frac{\partial L}{\partial z^{(\ell)}}$$

**Output layer** ($\ell = L$):
$$\delta^{(L)} = \frac{\partial L}{\partial z^{(L)}} = \hat{y} - y$$
(for softmax + cross-entropy)

**Hidden layers** ($\ell = L-1, \ldots, 1$):
$$\delta^{(\ell)} = (W^{(\ell+1)})^T \delta^{(\ell+1)} \odot \sigma'(z^{(\ell)})$$

where $\odot$ is element-wise product.

**Gradients** for parameters:
$$\begin{aligned}
\frac{\partial L}{\partial W^{(\ell)}} &= \delta^{(\ell)} (a^{(\ell-1)})^T \\
\frac{\partial L}{\partial b^{(\ell)}} &= \delta^{(\ell)}
\end{aligned}$$

### 3.3 Derivation of Hidden Layer Error

**Goal**: Derive $\delta^{(\ell)} = \frac{\partial L}{\partial z^{(\ell)}}$

**Chain rule**:
$$\frac{\partial L}{\partial z^{(\ell)}} = \frac{\partial L}{\partial z^{(\ell+1)}} \frac{\partial z^{(\ell+1)}}{\partial a^{(\ell)}} \frac{\partial a^{(\ell)}}{\partial z^{(\ell)}}$$

**Step 1**: $\frac{\partial a^{(\ell)}}{\partial z^{(\ell)}} = \sigma'(z^{(\ell)})$ (element-wise)

**Step 2**: $z^{(\ell+1)} = W^{(\ell+1)} a^{(\ell)} + b^{(\ell+1)}$

So: $\frac{\partial z^{(\ell+1)}}{\partial a^{(\ell)}} = W^{(\ell+1)}$

**Step 3**: By chain rule (vector-Jacobian product):
$$\frac{\partial L}{\partial a^{(\ell)}} = \left(\frac{\partial z^{(\ell+1)}}{\partial a^{(\ell)}}\right)^T \frac{\partial L}{\partial z^{(\ell+1)}} = (W^{(\ell+1)})^T \delta^{(\ell+1)}$$

**Final**:
$$\delta^{(\ell)} = \frac{\partial L}{\partial a^{(\ell)}} \odot \frac{\partial a^{(\ell)}}{\partial z^{(\ell)}} = (W^{(\ell+1)})^T \delta^{(\ell+1)} \odot \sigma'(z^{(\ell)})$$

✅ **QED**

In [ ]:
# Example 3: Full Network Backpropagation
print("Example 3: Multi-Layer Network\n" + "="*50)

class SimpleNetwork:
    """
    3-layer neural network with backpropagation
    """
    
    def __init__(self, input_dim, hidden1_dim, hidden2_dim, output_dim):
        # Initialize weights (Xavier initialization)
        self.W1 = np.random.randn(hidden1_dim, input_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros((hidden1_dim, 1))
        
        self.W2 = np.random.randn(hidden2_dim, hidden1_dim) * np.sqrt(2.0 / hidden1_dim)
        self.b2 = np.zeros((hidden2_dim, 1))
        
        self.W3 = np.random.randn(output_dim, hidden2_dim) * np.sqrt(2.0 / hidden2_dim)
        self.b3 = np.zeros((output_dim, 1))
        
        # Cache for backward pass
        self.cache = {}
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def sigmoid_derivative(self, z):
        s = self.sigmoid(z)
        return s * (1 - s)
    
    def softmax(self, z):
        # Stable softmax
        exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
        return exp_z / np.sum(exp_z, axis=0, keepdims=True)
    
    def forward(self, X):
        """
        Forward pass
        
        X: (input_dim, batch_size)
        """
        # Layer 1
        z1 = self.W1 @ X + self.b1
        a1 = self.sigmoid(z1)
        
        # Layer 2
        z2 = self.W2 @ a1 + self.b2
        a2 = self.sigmoid(z2)
        
        # Layer 3 (output)
        z3 = self.W3 @ a2 + self.b3
        y_hat = self.softmax(z3)
        
        # Cache for backward pass
        self.cache = {
            'X': X,
            'z1': z1, 'a1': a1,
            'z2': z2, 'a2': a2,
            'z3': z3, 'y_hat': y_hat
        }
        
        return y_hat
    
    def backward(self, y_true):
        """
        Backward pass (compute gradients)
        
        y_true: (output_dim, batch_size) - one-hot encoded
        """
        batch_size = y_true.shape[1]
        
        # Retrieve cached values
        X = self.cache['X']
        z1, a1 = self.cache['z1'], self.cache['a1']
        z2, a2 = self.cache['z2'], self.cache['a2']
        z3, y_hat = self.cache['z3'], self.cache['y_hat']
        
        # Output layer error (softmax + cross-entropy)
        delta3 = y_hat - y_true  # (output_dim, batch_size)
        
        # Hidden layer 2 error
        delta2 = (self.W3.T @ delta3) * self.sigmoid_derivative(z2)
        
        # Hidden layer 1 error
        delta1 = (self.W2.T @ delta2) * self.sigmoid_derivative(z1)
        
        # Compute gradients
        dW3 = (1 / batch_size) * (delta3 @ a2.T)
        db3 = (1 / batch_size) * np.sum(delta3, axis=1, keepdims=True)
        
        dW2 = (1 / batch_size) * (delta2 @ a1.T)
        db2 = (1 / batch_size) * np.sum(delta2, axis=1, keepdims=True)
        
        dW1 = (1 / batch_size) * (delta1 @ X.T)
        db1 = (1 / batch_size) * np.sum(delta1, axis=1, keepdims=True)
        
        return {
            'dW1': dW1, 'db1': db1,
            'dW2': dW2, 'db2': db2,
            'dW3': dW3, 'db3': db3
        }
    
    def cross_entropy_loss(self, y_hat, y_true):
        """Compute cross-entropy loss"""
        batch_size = y_true.shape[1]
        # Clip to avoid log(0)
        y_hat_clipped = np.clip(y_hat, 1e-10, 1 - 1e-10)
        loss = -np.sum(y_true * np.log(y_hat_clipped)) / batch_size
        return loss

# Test the network
np.random.seed(42)

# Create small network
net = SimpleNetwork(input_dim=3, hidden1_dim=4, hidden2_dim=4, output_dim=2)

# Create dummy data
X = np.random.randn(3, 5)  # 5 samples, 3 features
y = np.array([[1, 0, 1, 0, 1],
              [0, 1, 0, 1, 0]])  # One-hot encoded labels

print("Network Architecture:")
print(f"  Input: {net.W1.shape[1]}")
print(f"  Hidden 1: {net.W1.shape[0]}")
print(f"  Hidden 2: {net.W2.shape[0]}")
print(f"  Output: {net.W3.shape[0]}")
print()

# Forward pass
y_hat = net.forward(X)
loss = net.cross_entropy_loss(y_hat, y)

print(f"Loss: {loss:.6f}")
print()

# Backward pass
grads = net.backward(y)

print("Gradient shapes:")
for name, grad in grads.items():
    print(f"  {name}: {grad.shape}")

print("\n✅ Forward and backward pass complete!")

## 4. Gradient Checking

### 4.1 Why Gradient Checking?

Backpropagation is complex → easy to make mistakes!

**Gradient checking** verifies analytical gradients against numerical gradients.

### 4.2 Numerical Gradient

**Two-sided difference** (more accurate):
$$\frac{\partial L}{\partial \theta} \approx \frac{L(\theta + \epsilon) - L(\theta - \epsilon)}{2\epsilon}$$

Error: $O(\epsilon^2)$

**One-sided difference** (less accurate):
$$\frac{\partial L}{\partial \theta} \approx \frac{L(\theta + \epsilon) - L(\theta)}{\epsilon}$$

Error: $O(\epsilon)$

### 4.3 Relative Error

$$\text{relative error} = \frac{\|\nabla_{\text{analytical}} - \nabla_{\text{numerical}}\|_2}{\|\nabla_{\text{analytical}}\|_2 + \|\nabla_{\text{numerical}}\|_2}$$

**Guidelines**:
- Relative error < 1e-7: ✅ Perfect!
- Relative error < 1e-4: ⚠️ Acceptable (might have issues)
- Relative error > 1e-4: ❌ Likely wrong implementation

In [ ]:
# Example 4: Gradient Checking
print("Example 4: Gradient Checking\n" + "="*50)

def gradient_check(net, X, y, epsilon=1e-5):
    """
    Check gradients using numerical differentiation
    """
    # Get analytical gradients
    y_hat = net.forward(X)
    analytical_grads = net.backward(y)
    
    # Parameters to check
    params = {
        'W1': net.W1, 'b1': net.b1,
        'W2': net.W2, 'b2': net.b2,
        'W3': net.W3, 'b3': net.b3
    }
    
    results = {}
    
    for param_name, param in params.items():
        analytical = analytical_grads[f'd{param_name}']
        numerical = np.zeros_like(param)
        
        # Compute numerical gradient for each element
        it = np.nditer(param, flags=['multi_index'])
        
        count = 0
        total = param.size
        
        while not it.finished:
            idx = it.multi_index
            
            # Save original value
            old_value = param[idx]
            
            # Compute f(theta + epsilon)
            param[idx] = old_value + epsilon
            y_hat_plus = net.forward(X)
            loss_plus = net.cross_entropy_loss(y_hat_plus, y)
            
            # Compute f(theta - epsilon)
            param[idx] = old_value - epsilon
            y_hat_minus = net.forward(X)
            loss_minus = net.cross_entropy_loss(y_hat_minus, y)
            
            # Numerical gradient
            numerical[idx] = (loss_plus - loss_minus) / (2 * epsilon)
            
            # Restore original value
            param[idx] = old_value
            
            it.iternext()
            count += 1
            
            # Print progress for large matrices
            if count % max(1, total // 10) == 0:
                print(f"  {param_name}: {count}/{total} elements checked", end='\r')
        
        # Compute relative error
        numerator = np.linalg.norm(analytical - numerical)
        denominator = np.linalg.norm(analytical) + np.linalg.norm(numerical)
        relative_error = numerator / (denominator + 1e-10)
        
        results[param_name] = {
            'relative_error': relative_error,
            'max_diff': np.max(np.abs(analytical - numerical))
        }
        
        print(f"  {param_name}: ✓ Complete                    ")
    
    return results

# Run gradient check (on small network)
print("Running gradient check...\n")

net_small = SimpleNetwork(input_dim=2, hidden1_dim=3, hidden2_dim=3, output_dim=2)
X_small = np.random.randn(2, 3)
y_small = np.array([[1, 0, 1], [0, 1, 0]])

results = gradient_check(net_small, X_small, y_small)

print("\nGradient Check Results:")
print("="*60)
print(f"{'Parameter':<10} {'Relative Error':<20} {'Max Abs Diff':<15} {'Status'}")
print("="*60)

for param_name, result in results.items():
    rel_err = result['relative_error']
    max_diff = result['max_diff']
    
    if rel_err < 1e-7:
        status = "✅ Perfect"
    elif rel_err < 1e-4:
        status = "⚠️  Acceptable"
    else:
        status = "❌ Error"
    
    print(f"{param_name:<10} {rel_err:<20.2e} {max_diff:<15.2e} {status}")

print("="*60)
print("\n✅ Gradient checking complete!")

## 5. Layer-Specific Gradients

### 5.1 Fully Connected Layer

**Forward**:
$$z = Wx + b$$

**Backward**:
$$\begin{aligned}
\frac{\partial L}{\partial W} &= \frac{\partial L}{\partial z} x^T \\
\frac{\partial L}{\partial b} &= \frac{\partial L}{\partial z} \\
\frac{\partial L}{\partial x} &= W^T \frac{\partial L}{\partial z}
\end{aligned}$$

### 5.2 ReLU Activation

**Forward**:
$$a = \max(0, z)$$

**Derivative**:
$$\frac{\partial a}{\partial z} = \begin{cases}
1 & \text{if } z > 0 \\
0 & \text{if } z \leq 0
\end{cases}$$

**Backward**:
$$\frac{\partial L}{\partial z} = \frac{\partial L}{\partial a} \odot \mathbb{1}_{z > 0}$$

### 5.3 Softmax + Cross-Entropy

**Forward**:
$$\begin{aligned}
\hat{y}_i &= \frac{e^{z_i}}{\sum_j e^{z_j}} \\
L &= -\sum_i y_i \log \hat{y}_i
\end{aligned}$$

**Backward** (amazing simplification!):
$$\frac{\partial L}{\partial z} = \hat{y} - y$$

**Proof**:
$$\frac{\partial L}{\partial z_i} = \sum_j \frac{\partial L}{\partial \hat{y}_j} \frac{\partial \hat{y}_j}{\partial z_i}$$

For $j = i$:
$$\frac{\partial \hat{y}_i}{\partial z_i} = \hat{y}_i(1 - \hat{y}_i)$$

For $j \neq i$:
$$\frac{\partial \hat{y}_j}{\partial z_i} = -\hat{y}_j \hat{y}_i$$

And $\frac{\partial L}{\partial \hat{y}_j} = -y_j / \hat{y}_j$, so:

$$\frac{\partial L}{\partial z_i} = -\frac{y_i}{\hat{y}_i} \hat{y}_i(1-\hat{y}_i) + \sum_{j \neq i} \left(-\frac{y_j}{\hat{y}_j}\right)(-\hat{y}_j\hat{y}_i)$$
$$= -y_i + y_i\hat{y}_i + \sum_{j \neq i} y_j \hat{y}_i$$
$$= -y_i + \hat{y}_i \sum_j y_j$$
$$= -y_i + \hat{y}_i \quad \text{(since } \sum_j y_j = 1\text{)}$$
$$= \hat{y}_i - y_i$$

✅ **QED**

This is why softmax + cross-entropy is preferred!

---

## Summary

### Key Takeaways

1. **Backpropagation is just the chain rule** applied systematically
2. **Computational graphs** make the process clear and automatic
3. **Error terms** $\delta^{(\ell)}$ propagate backward through layers
4. **Always verify** your gradients with numerical differentiation
5. **Softmax + cross-entropy** has beautiful gradient: $\hat{y} - y$

### Common Pitfalls

❌ Forgetting to transpose matrices  
❌ Wrong dimensions in matrix multiplication  
❌ Not using element-wise multiplication where needed  
❌ Sign errors in gradients  
❌ Not averaging over batch

### Best Practices

✅ Always check gradient dimensions  
✅ Use gradient checking during development  
✅ Cache intermediate values in forward pass  
✅ Use stable implementations (softmax, etc.)  
✅ Test on small examples first

---

## References

1. **Rumelhart, D. E., Hinton, G. E., & Williams, R. J.** (1986). *Learning representations by back-propagating errors*. Nature, 323(6088), 533-536.

2. **Goodfellow, I., Bengio, Y., & Courville, A.** (2016). *Deep Learning*. MIT Press.
   - Chapter 6: Deep Feedforward Networks

3. **Nielsen, M. A.** (2015). *Neural Networks and Deep Learning*. Determination Press.
   - Chapter 2: How the backpropagation algorithm works

4. **CS231n Stanford**: Backpropagation Notes
   - http://cs231n.github.io/optimization-2/

---

**Next Notebook**: [Convolutional Networks](./Conv_Networks_Mathematics.ipynb)